In [ ]:
import pandas as pd
import numpy as np
import itertools

from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import f1_score

In [ ]:
# Evaluated models
bert_models = [
    "distilbert-base-spanish-uncased",
    "roberta-large-bne",
    "mdeberta-v3-base"
    # "longformer-base-4096"
]
open_source_llms = [
    "llama-3.1-8b",
    "gpt-oss-20b",
    "deepseek-chat"
]
proprietary_llms = [
    "gpt-5.2",
    "gpt-5-mini",
    "claude-sonnet-4-6",
    "claude-haiku-4.5",
    "gemini-3.1-pro-preview",
    "gemini-3-flash-preview"
]

llm_models = open_source_llms + proprietary_llms
models = bert_models + llm_models

tasks = ["anx", "dep", "ed", "multi"]

# Read metrics tables

In [ ]:
# Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
base = "/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
CACHE_DIR = base + "metricas"

# Read data and tables generated in 5_1_metrics.ipynb
data = pd.read_parquet(f"{CACHE_DIR}/all_predictions.parquet")
metricas_anx = pd.read_csv(f"{CACHE_DIR}/metricas_anx.csv", index_col="model")
metricas_dep = pd.read_csv(f"{CACHE_DIR}/metricas_dep.csv", index_col="model")
metricas_ed = pd.read_csv(f"{CACHE_DIR}/metricas_ed.csv", index_col="model")
metricas_multi = pd.read_csv(f"{CACHE_DIR}/metricas_multi.csv", index_col="model")

In [ ]:
data.tail()

,sample_id,user_id,model,task,y_true,y_pred
4811,subject68_ed,subject68,gemini-3-flash-preview,multi,3,3
4812,subject8_ed,subject8,gemini-3-flash-preview,multi,3,3
4813,subject83_ed,subject83,gemini-3-flash-preview,multi,0,2
4814,subject95_ed,subject95,gemini-3-flash-preview,multi,3,3
4815,subject97_ed,subject97,gemini-3-flash-preview,multi,3,3


# Approximate Randomization Test (ART)

The Approximate Randomization Test (ART) is a non-parametric test widely used in NLP to determine if the observed performance difference between two models is truly significant or could be due to chance.

The main idea of the method consists of initially assuming the null hypothesis that both models are equivalent. If this were true, then the predictions of both systems would be interchangeable without substantially altering the overall performance. To verify this, the test generates thousands of random permutations by swapping the predictions of both models sample by sample and recalculating the macro F1 difference in each iteration.

Let $\Delta_{obs}$ be the observed difference between both models:

$$
\Delta_{obs} = M_A - M_B
$$

where $M_A$ and $M_B$ represent the macro F1 value obtained by each system.

Next, we calculate how many of the random permutations produce a difference equal to or greater than the originally observed one. The p-value is estimated as:

$$
p = \frac{c + 1}{N + 1}
$$

where:

$c$ is the number of permutations whose difference is at least as extreme as $|\Delta_{obs}|$.
$N$ is the total number of iterations performed.

A small p-value indicates that it is unlikely to obtain a difference like the observed one simply by chance, suggesting that one of the models actually has better performance.

In this work, 10,000 iterations were used to estimate the p-values associated with each comparison.

In [ ]:
def f1_macro_metric(y_true, y_pred, task):
    """
    Macro F1 with fixed labels according to the task.
    """

    if task == "multi":
        labels = [0, 1, 2, 3]
    else:
        labels = [0, 1]

    return f1_score(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0
    )

In [ ]:
def get_paired_data(data, model_a, model_b, task):
    """
    Obtains aligned predictions using only
    the sample_ids present in both models.
    """

    df_a = data[
        (data["model"] == model_a) &
        (data["task"] == task)
    ][["sample_id", "y_true", "y_pred"]]

    df_b = data[
        (data["model"] == model_b) &
        (data["task"] == task)
    ][["sample_id", "y_pred"]]

    # Remove NaNs before merge
    df_a = df_a.dropna()
    df_b = df_b.dropna()

    # REAL intersection
    common_ids = (
        set(df_a["sample_id"])
        &
        set(df_b["sample_id"])
    )

    df_a = df_a[df_a["sample_id"].isin(common_ids)]
    df_b = df_b[df_b["sample_id"].isin(common_ids)]

    # Aligned merge
    df = (
        df_a
        .merge(
            df_b,
            on="sample_id",
            suffixes=("_a", "_b")
        )
        .sort_values("sample_id")
    )

    return (
        df["y_true"],
        df["y_pred_a"],
        df["y_pred_b"]
    )

In [ ]:
def approximate_randomization_test(
    y_true,
    y_pred_a,
    y_pred_b,
    metric_fn,
    task,
    n_iterations=10000,
    random_state=42
):
    """
    Approximate Randomization Test (ART)
    to compare two models using macro F1.

    Null hypothesis:
    both models have the same performance.
    """

    rng = np.random.default_rng(random_state)

    # REAL observed difference
    observed_diff = abs(
        metric_fn(y_true, y_pred_a, task)
        -
        metric_fn(y_true, y_pred_b, task)
    )

    count = 0

    y_pred_a = np.array(y_pred_a)
    y_pred_b = np.array(y_pred_b)

    for _ in range(n_iterations):

        # Copies
        swap_a = y_pred_a.copy()
        swap_b = y_pred_b.copy()

        # Random mask
        mask = rng.random(len(y_true)) < 0.5

        # Swap predictions
        swap_a[mask], swap_b[mask] = (
            swap_b[mask],
            swap_a[mask]
        )

        # New difference
        new_diff = abs(
            metric_fn(y_true, swap_a, task)
            -
            metric_fn(y_true, swap_b, task)
        )

        if new_diff >= observed_diff:
            count += 1

    # p-value
    p_value = (count + 1) / (n_iterations + 1)

    return p_value

# McNemar Test

The McNemar test is a non-parametric statistical test designed to compare two classifiers evaluated on exactly the same set of examples. Unlike ART, McNemar does not directly compare aggregated metrics like F1-score, but analyzes the disagreements between both models at the individual prediction level.

The fundamental idea is to check if one of the models corrects errors that the other is unable to solve. For this, only cases where both models disagree are considered.

The test is based on the following contingency table:

$$
\begin{array}{c|cc}
 & B\ \text{correct} & B\ \text{incorrect} \\
\hline
A\ \text{correct} & n_{11} & n_{10} \\
A\ \text{incorrect} & n_{01} & n_{00}
\end{array}
$$

where:

$n_{10}$ represents the examples correctly classified by A but not by B.
$n_{01}$ represents the examples correctly classified by B but not by A.

The null hypothesis states that both models have a similar probability of making different errors, that is:

$$
n_{10} \approx n_{01}
$$

If one of the models systematically guesses examples that the other fails, the test will produce a small p-value, indicating that the difference between both classifiers is probably not due to chance.

In this work, the exact version of the McNemar test was used, which is especially suitable for small sample sizes.

In [ ]:
def mcnemar_test(y_true, y_pred_a, y_pred_b):
    """
    Exact McNemar test for binary classification.
    """

    correct_a = (y_pred_a == y_true)
    correct_b = (y_pred_b == y_true)

    n01 = np.sum((correct_a == 1) & (correct_b == 0))
    n10 = np.sum((correct_a == 0) & (correct_b == 1))

    table = [
        [0, n01],
        [n10, 0]
    ]

    result = mcnemar(
        table,
        exact=True
    )

    return result.pvalue

# Combined ART + McNemar table

In [ ]:
def bootstrap_delta_ci(
    y_true,
    y_a,
    y_b,
    task,
    n_bootstrap=5000,
    confidence_level=0.95,
    seed=42
):
    """
    Percentile bootstrap CI for the macro F1 difference between two models.
    """

    rng = np.random.default_rng(seed)

    y_true = np.asarray(y_true)
    y_a = np.asarray(y_a)
    y_b = np.asarray(y_b)

    n = len(y_true)

    deltas = np.empty(n_bootstrap)

    for i in range(n_bootstrap):

        idx = rng.choice(n, size=n, replace=True)

        deltas[i] = (
            f1_macro_metric(
                y_true[idx],
                y_a[idx],
                task
            )
            -
            f1_macro_metric(
                y_true[idx],
                y_b[idx],
                task
            )
        )

    alpha = (1 - confidence_level) / 2

    return (
        np.quantile(deltas, alpha),
        np.quantile(deltas, 1 - alpha)
    )

In [ ]:
def format_pvalue(p):
    """
    Formats an already corrected p-value.
    """
    if p < 0.001:
        return "<0.001***"
    elif p < 0.01:
        return f"{p:.3f}**"
    elif p < 0.05:
        return f"{p:.3f}*"
    else:
        return f"{p:.3f}"

In [ ]:
def paired_comparison_table_art_mcnemar(models, task):

    rows = []

    art_pvalues = []
    mcnemar_pvalues = []
    mcnemar_idx = []

    for model_a, model_b in itertools.combinations(models, 2):

        y_true, y_a, y_b = get_paired_data(
            data,
            model_a,
            model_b,
            task
        )

        # Number of common samples used for comparison
        n_samples = len(y_true)

        # Mean delta
        diff_mean = (
            f1_macro_metric(y_true, y_a, task)
            -
            f1_macro_metric(y_true, y_b, task)
        )

        ci_low, ci_high = bootstrap_delta_ci(
            y_true,
            y_a,
            y_b,
            task
        )

        # =====================================================
        # Approximate Randomization Test
        # =====================================================

        art_p = approximate_randomization_test(
            y_true,
            y_a,
            y_b,
            f1_macro_metric,
            task,
            n_iterations=10000
        )

        art_pvalues.append(art_p)

        # =====================================================
        # McNemar
        # =====================================================

        if task != "multi":

            mcnemar_p = mcnemar_test(
                np.array(y_true),
                np.array(y_a),
                np.array(y_b)
            )

            mcnemar_pvalues.append(mcnemar_p)
            mcnemar_idx.append(len(rows))

        else:
            mcnemar_p = None

        rows.append({
            "Model A": model_a,
            "Model B": model_b,
            "Samples": n_samples,
            "ΔF1 (95% CI)":
                f"{diff_mean:.3f} "
                f"[{ci_low:.3f}, {ci_high:.3f}]",
            "ART p": art_p,
            "McNemar p": mcnemar_p
        })

    # =====================================================
    # Benjamini-Hochberg (ART)
    # =====================================================

    _, art_adj, _, _ = multipletests(
        art_pvalues,
        alpha=0.05,
        method="fdr_bh"
    )

    for row, p in zip(rows, art_adj):
        row["ART"] = format_pvalue(p)

    # =====================================================
    # Benjamini-Hochberg (McNemar)
    # =====================================================

    if task != "multi":
        _, mcnemar_adj, _, _ = multipletests(
            mcnemar_pvalues,
            alpha=0.05,
            method="fdr_bh"
        )

        for idx, p in zip(mcnemar_idx, mcnemar_adj):
            rows[idx]["McNemar"] = format_pvalue(p)

    else:
        for row in rows:
            row["McNemar"] = "--"

    # Remove unformatted p-values
    for row in rows:
        row.pop("ART p")
        row.pop("McNemar p")

    return pd.DataFrame(rows)

In [ ]:
def get_top_models(table_ci, metric="f1_macro"):
    """
    Selects the models to be compared in the paired bootstrap:
    - best BERT model
    - best open-source LLM
    - two best proprietary LLMs
    """
    scores = table_ci[metric]

    best_bert = scores.loc[scores.index.intersection(bert_models)].idxmax()

    best_open_llm = scores.loc[
        scores.index.intersection(open_source_llms)
    ].idxmax()

    best_proprietary = (
        scores.loc[
            scores.index.intersection(proprietary_llms)
        ]
        .sort_values(ascending=False)
        .head(2)
        .index
        .tolist()
    )

    return [best_bert, best_open_llm] + best_proprietary

In [ ]:
def display_art_mcnemar(metrics_df, task):

    top_models = get_top_models(metrics_df)

    comparison_table = paired_comparison_table_art_mcnemar(
        top_models,
        task
    )

    display(comparison_table)

In [ ]:
display_art_mcnemar(metricas_anx, "anx")

,Model A,Model B,Samples,ΔF1 (95% CI),ART,McNemar
0,roberta-large-bne,gpt-oss-20b,75,"-0.220 [-0.401, -0.052]",0.071,0.067
1,roberta-large-bne,claude-sonnet-4-6,75,"-0.208 [-0.416, 0.011]",0.163,0.067
2,roberta-large-bne,gemini-3-flash-preview,75,"-0.162 [-0.371, 0.057]",0.317,0.077
3,gpt-oss-20b,claude-sonnet-4-6,75,"0.012 [-0.091, 0.129]",1.000,1.000
4,gpt-oss-20b,gemini-3-flash-preview,75,"0.058 [-0.068, 0.228]",0.752,1.000
5,claude-sonnet-4-6,gemini-3-flash-preview,75,"0.046 [0.000, 0.173]",1.000,1.000


In [ ]:
display_art_mcnemar(metricas_dep, "dep")

,Model A,Model B,Samples,ΔF1 (95% CI),ART,McNemar
0,roberta-large-bne,deepseek-chat,75,"-0.013 [-0.115, 0.090]",1.000,1.000
1,roberta-large-bne,gemini-3.1-pro-preview,73,"-0.015 [-0.126, 0.096]",1.000,1.000
2,roberta-large-bne,gemini-3-flash-preview,73,"-0.002 [-0.131, 0.131]",1.000,1.000
3,deepseek-chat,gemini-3.1-pro-preview,73,"-0.002 [-0.122, 0.118]",1.000,1.000
4,deepseek-chat,gemini-3-flash-preview,73,"0.012 [-0.117, 0.141]",1.000,1.000
5,gemini-3.1-pro-preview,gemini-3-flash-preview,73,"0.013 [-0.063, 0.101]",1.000,1.000


In [ ]:
display_art_mcnemar(metricas_ed, "ed")

,Model A,Model B,Samples,ΔF1 (95% CI),ART,McNemar
0,roberta-large-bne,gpt-oss-20b,51,"-0.020 [-0.098, 0.043]",1.000,1.000
1,roberta-large-bne,gpt-5.2,51,"-0.040 [-0.103, 0.000]",1.000,1.000
2,roberta-large-bne,claude-sonnet-4-6,51,"-0.020 [-0.098, 0.043]",1.000,1.000
3,gpt-oss-20b,gpt-5.2,51,"-0.020 [-0.065, 0.000]",1.000,1.000
4,gpt-oss-20b,claude-sonnet-4-6,51,"0.000 [-0.060, 0.060]",1.000,1.000
5,gpt-5.2,claude-sonnet-4-6,51,"0.020 [0.000, 0.064]",1.000,1.000


In [ ]:
display_art_mcnemar(metricas_multi, "multi")

,Model A,Model B,Samples,ΔF1 (95% CI),ART,McNemar
0,roberta-large-bne,deepseek-chat,201,"0.076 [0.009, 0.147]",0.060,--
1,roberta-large-bne,gemini-3-flash-preview,199,"-0.009 [-0.068, 0.051]",0.917,--
2,roberta-large-bne,gemini-3.1-pro-preview,199,"-0.007 [-0.066, 0.052]",0.917,--
3,deepseek-chat,gemini-3-flash-preview,199,"-0.086 [-0.146, -0.028]",0.017*,--
4,deepseek-chat,gemini-3.1-pro-preview,199,"-0.084 [-0.146, -0.027]",0.017*,--
5,gemini-3-flash-preview,gemini-3.1-pro-preview,199,"0.002 [-0.030, 0.033]",0.917,--


# Intra-family comparison (large vs light model)

In [ ]:
def family_comparison_table_art_mcnemar(models, task):
    """
    Generates an intra-family statistical comparison table
    between pairs of models (large vs light).

    The function assumes that the models are provided sorted
    in consecutive pairs.
    """

    rows = []

    art_pvalues = []
    mcnemar_pvalues = []
    mcnemar_idx = []

    # =====================================================
    # Intra-family comparisons
    # =====================================================

    for i in range(0, len(models) - 1, 2):

        model_a = models[i]
        model_b = models[i + 1]

        y_true, y_a, y_b = get_paired_data(
            data,
            model_a,
            model_b,
            task
        )

        # Number of common samples
        n_samples = len(y_true)

        # Macro F1 difference
        diff_mean = (
            f1_macro_metric(y_true, y_a, task)
            -
            f1_macro_metric(y_true, y_b, task)
        )

        # Bootstrap CI of ΔF1
        ci_low, ci_high = bootstrap_delta_ci(
            y_true,
            y_a,
            y_b,
            task
        )

        # =====================================================
        # Approximate Randomization Test
        # =====================================================

        art_p = approximate_randomization_test(
            y_true,
            y_a,
            y_b,
            f1_macro_metric,
            task,
            n_iterations=10000
        )

        art_pvalues.append(art_p)

        # =====================================================
        # McNemar
        # =====================================================

        if task != "multi":

            mcnemar_p = mcnemar_test(
                np.array(y_true),
                np.array(y_a),
                np.array(y_b)
            )

            mcnemar_pvalues.append(mcnemar_p)
            mcnemar_idx.append(len(rows))

        else:

            mcnemar_p = None

        rows.append({
            "Model A": model_a,
            "Model B": model_b,
            "Samples": n_samples,
            "ΔF1 (95% CI)":
                f"{diff_mean:.3f} "
                f"[{ci_low:.3f}, {ci_high:.3f}]",
            "ART p": art_p,
            "McNemar p": mcnemar_p
        })

    # =====================================================
    # Benjamini-Hochberg (ART)
    # =====================================================

    _, art_adj, _, _ = multipletests(
        art_pvalues,
        alpha=0.05,
        method="fdr_bh"
    )

    for row, p in zip(rows, art_adj):
        row["ART"] = format_pvalue(p)

    # =====================================================
    # Benjamini-Hochberg (McNemar)
    # =====================================================

    if task != "multi":

        _, mcnemar_adj, _, _ = multipletests(
            mcnemar_pvalues,
            alpha=0.05,
            method="fdr_bh"
        )

        for idx, p in zip(mcnemar_idx, mcnemar_adj):
            rows[idx]["McNemar"] = format_pvalue(p)

    else:

        for row in rows:
            row["McNemar"] = "--"

    # =====================================================
    # Remove unformatted p-values
    # =====================================================

    for row in rows:
        row.pop("ART p")
        row.pop("McNemar p")

    return pd.DataFrame(rows)

In [ ]:
family_comparison_table_art_mcnemar(proprietary_llms, "anx")

,Model A,Model B,Samples,ΔF1 (95% CI),ART,McNemar
0,gpt-5.2,gpt-5-mini,75,"0.000 [0.000, 0.000]",1.000,1.000
1,claude-sonnet-4-6,claude-haiku-4.5,75,"0.099 [0.000, 0.287]",0.760,0.750
2,gemini-3.1-pro-preview,gemini-3-flash-preview,75,"-0.053 [-0.189, 0.087]",0.760,0.750


In [ ]:
family_comparison_table_art_mcnemar(proprietary_llms, "dep")

,Model A,Model B,Samples,ΔF1 (95% CI),ART,McNemar
0,gpt-5.2,gpt-5-mini,75,"-0.006 [-0.117, 0.106]",0.898,1.000
1,claude-sonnet-4-6,claude-haiku-4.5,75,"0.083 [-0.016, 0.192]",0.298,1.000
2,gemini-3.1-pro-preview,gemini-3-flash-preview,73,"0.013 [-0.063, 0.101]",0.898,1.000


In [ ]:
family_comparison_table_art_mcnemar(proprietary_llms, "ed")

,Model A,Model B,Samples,ΔF1 (95% CI),ART,McNemar
0,gpt-5.2,gpt-5-mini,51,"0.079 [0.020, 0.160]",0.394,0.375
1,claude-sonnet-4-6,claude-haiku-4.5,51,"0.039 [0.000, 0.099]",0.504,0.500
2,gemini-3.1-pro-preview,gemini-3-flash-preview,51,"0.058 [-0.020, 0.152]",0.504,0.500


In [ ]:
family_comparison_table_art_mcnemar(proprietary_llms, "multi")

,Model A,Model B,Samples,ΔF1 (95% CI),ART,McNemar
0,gpt-5.2,gpt-5-mini,201,"0.007 [-0.042, 0.057]",0.917,--
1,claude-sonnet-4-6,claude-haiku-4.5,201,"-0.038 [-0.091, 0.012]",0.462,--
2,gemini-3.1-pro-preview,gemini-3-flash-preview,199,"-0.002 [-0.033, 0.030]",0.917,--
